In [ ]:
# ------------------------------------
# OLD CODE
# ------------------------------------
# import glob
# import numpy as np
# import librosa

# segment_dur = 2
# directories = ['guitar', 'non_guitar']
# def load_segments(directory):
#     segments = []
#     for path in glob.glob(f'{directory}/*.wav'):
#         signal, sr = librosa.load(path, sr=None, mono=True)
#         seg_len = sr * segment_dur
#         audio_segments = librosa.util.frame(signal, frame_length=seg_len, hop_length=seg_len, axis=0)
#         segments.append(audio_segments)
#     return np.concatenate(segments)

# guitar_segments = load_segments('guitar')
# non_guitar_segments = load_segments('non_guitar')
# def get_chords(path):

#     chords = []
#     with open(path) as f:
#         for line in f:      
#             start_s, end_s, chord = line.strip().split('\t')
#             chords.append(chord)
#     return chords


# guitar_chords = []
# for i in range(6):
#     guitar_chords += get_chords(f'guitar/guitar_annotation.lab')

# non_guitar_chords = []
# for i in range(10):
#     non_guitar_chords += get_chords(f'non_guitar/non_guitar_annotation.lab')

using mps


In [ ]:
#stuff needed to use a trained model (the presentation needs these too)
import torch

#set up device
device = "mps" if torch.backends.mps.is_available() else "cpu" #mps is the mac's gpu
print("using", device)


#gets the predicted labels and confidence for the sklearn models
def predict_sklearn_model(model, data):
    label = model.predict(data)
    confidence = None #a plain SVM cant give confidence
    if hasattr(model, "predict_proba"):
        confidence = model.predict_proba(data).max(axis=1)
    else:
        confidence = 1
    return label, confidence


#scales the data and turns it into a tensor for the MLP and CNN
def get_inputs(data, scaler):
    if scaler is not None:
        data = scaler.transform(data)
    return torch.tensor(data, dtype=torch.float32).to(device)


#gets the predicted labels and confidence for the MLP and CNN
def predict_net(net, data):
    inputs = get_inputs(data, net["scaler"])
    net["model"].eval() #puts the model in testing mode
    with torch.no_grad(): #dont need gradients when im only predicting
        scores = net["model"](inputs)
    probabilities = torch.softmax(scores, dim=1).cpu().numpy() #turns the scores into probabilities that add up to 1
    labels = net["classes"][probabilities.argmax(axis=1)] #picks the label with the highest probability
    confidences = probabilities.max(axis=1)
    return labels, confidences


#joins the root and quality labels back together
def combine_labels(roots, qualities):
    return [root + ":" + quality for root, quality in zip(roots, qualities)]

using mps


In [3]:
#Set up and helper functions for training
import copy
import librosa
import numpy as np
import pandas as pd
import torch.nn as nn

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, Normalizer, StandardScaler
from sklearn.svm import SVC

#Helpers

#makes a fresh untrained sklearn model with its best tuned settings
def build_sklearn_model(name, target):
    model = clone(sklearn_models[name])
    return model.set_params(**best_params.get((name, target), {}))


#the MLP uses X and the CNN uses the picture
def get_data(name):
    if name == "MLP":
        return X
    return picture


#Set up

#Generate labels

#True keeps the inversions (288 chords), False drops them (84 chords)
INCLUDE_INVERSIONS = True 
labels = pd.read_csv("data/processed/labels.csv")

if INCLUDE_INVERSIONS:
    y = labels["original_label"].to_numpy()
else:
    y = labels["original_label"].str.split("/").str[0].to_numpy()


#Set up targets. Quality includes inversions.
targets = {
    "chord": y,
    "root": np.array([label.split(":")[0] for label in y]),
    "quality": np.array([label.split(":")[1] for label in y]),
}

for name, values in targets.items():
    print(name, len(set(values)), "classes")

chord 288 classes
root 12 classes
quality 24 classes


In [ ]:
#SPLIT OPTION 1: random (70% train, 15% validation, 15% test)
#only run one of the 2 split cells

all_idx = np.arange(len(y))
train_idx, rem_idx = train_test_split(all_idx, test_size=0.3, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(rem_idx, test_size=0.5, stratify=y[rem_idx], random_state=42)

split_used = "random"
print(split_used, len(train_idx), len(val_idx), len(test_idx))

[   0    1    2 ... 7395 7396 7397]
random 5178 1110 1110


In [ ]:
#SPLIT OPTION 2: by instrument (11 train, 2 validation, 3 test)

VAL_INSTRUMENTS = [4, 10]
TEST_INSTRUMENTS = [3, 6, 13]

instruments = labels["instrument"].unique()


val_mask = []
for val_instrument in VAL_INSTRUMENTS:
    val_mask.append(labels["instrument"] == instruments[val_instrument])
val_mask = np.array(val_mask).any(axis=0)

test_mask = []
for test_instrument in TEST_INSTRUMENTS:
    test_mask.append(labels["instrument"] == instruments[test_instrument])
test_mask = np.array(test_mask).any(axis=0)

train_idx = np.where(~val_mask & ~test_mask)[0]
val_idx = np.where(val_mask)[0]
test_idx = np.where(test_mask)[0]


split_used = "instrument"
print(split_used, len(train_idx), len(val_idx), len(test_idx))

instrument 5124 849 1425


In [ ]:
#FEATURES OPTION 1: chroma
#only run one of the 2 features cells

sr = 16000
X_audio = np.load("data/processed/clips_16k.npy").astype(np.float32)
picture = librosa.feature.chroma_stft(y=X_audio, sr=sr, n_fft=8192, hop_length=4096) # 7398 chords, 12 bins, 8 time frames
X = np.concatenate([picture.mean(axis=-1), picture.std(axis=-1)], axis=1) # 24 features: Mean and std of each of the 12 chroma bins

features_used = "chroma"
print(features_used, X.shape, picture.shape)

(7398, 32000)
chroma (7398, 24) (7398, 12, 8)


In [7]:
#FEATURES OPTION 2: wide spectrum (keeps the octaves separate, so inversions can be told apart)

sr = 16000
X_audio = np.load("data/processed/clips_16k.npy").astype(np.float32)

# Take abs of cqt to get magnitude of complex number (only want strength not phase)
cqt = np.abs(librosa.cqt(y=X_audio, sr=sr, hop_length=4096, fmin=librosa.note_to_hz("C1"), n_bins=84)) #84 notes, 8 time frames
picture = cqt / cqt.max(axis=(1, 2), keepdims=True) # 7398 chords, 84 bins, 8 time frames
X = X = np.concatenate([picture.mean(axis=-1), picture.std(axis=-1)], axis=1)
features_used = "wide spectrum"
print(features_used, X.shape, picture.shape)

wide spectrum (7398, 168) (7398, 84, 8)


In [46]:
#the starting settings for the sklearn models

#each model gets its own scaler, which only ever learns from the training data
sklearn_models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=500))]),
    "k-NN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
    "SVM": Pipeline([("scaler", StandardScaler()), ("model", SVC(C=10))]),
    "Random Forest": Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(random_state=42))]),
}

In [ ]:
#sklearn tuning, tries every combination in each grid and keeps whichever does best on the validation set

# ADD EVALUATION METRICS AND GRAPH STUFF HERE:

SCALERS = ["passthrough", StandardScaler(), MinMaxScaler(), Normalizer()] #passthrough means no scaling
grids = {
    "Logistic Regression": {"scaler": SCALERS, "model__C": [0.1, 1, 10]},
    "k-NN": {"scaler": SCALERS, "model__n_neighbors": [1, 3, 5, 9, 15, 25, 50, 75, 100], "model__metric": ["euclidean", "manhattan", "cosine"]},
    "SVM": {"scaler": SCALERS, "model__kernel": ["linear", "rbf", "poly"], "model__C": [0.1, 1, 10,100]},
    "Random Forest": {"scaler": ["passthrough", Normalizer()], "model__n_estimators": [100, 200, 300, 400, 500], "model__min_samples_leaf": [1, 3, 5, 10]},
}

best_params = {} 

for target in targets:
    for name, grid in grids.items():
        search = GridSearchCV(sklearn_models[name], grid, cv=[(train_idx, val_idx)], n_jobs=-1)
        search.fit(X, targets[target])
        best_params[name, target] = search.best_params_
        print(target, name, search.best_params_)

chord, Logistic Regression: val accuracy 0.402 {'model__C': 1, 'scaler': StandardScaler()}
    runner up 0.396 {'model__C': 10, 'scaler': MinMaxScaler()}
    runner up 0.393 {'model__C': 10, 'scaler': 'passthrough'}
    runner up 0.382 {'model__C': 10, 'scaler': Normalizer()}
chord, k-NN: val accuracy 0.519 {'model__metric': 'euclidean', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.519 {'model__metric': 'euclidean', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.519 {'model__metric': 'cosine', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.435 {'model__metric': 'manhattan', 'model__n_neighbors': 1, 'scaler': Normalizer()}
chord, SVM: val accuracy 0.538 {'model__C': 100, 'model__kernel': 'rbf', 'scaler': Normalizer()}
    runner up 0.534 {'model__C': 10, 'model__kernel': 'rbf', 'scaler': Normalizer()}
    runner up 0.523 {'model__C': 10, 'model__kernel': 'poly', 'scaler': Normalizer()}
    runner up 0.516 {'model__C': 100, 'model__

/Users/joanmk/UniStuff/IAML/IDMT-SMT-CHORDS/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


chord, Random Forest: val accuracy 0.516 {'model__min_samples_leaf': 3, 'model__n_estimators': 500, 'scaler': 'passthrough'}
    runner up 0.514 {'model__min_samples_leaf': 3, 'model__n_estimators': 400, 'scaler': 'passthrough'}
    runner up 0.511 {'model__min_samples_leaf': 1, 'model__n_estimators': 500, 'scaler': 'passthrough'}
    runner up 0.509 {'model__min_samples_leaf': 3, 'model__n_estimators': 300, 'scaler': 'passthrough'}
root, Logistic Regression: val accuracy 0.779 {'model__C': 10, 'scaler': Normalizer()}
    runner up 0.759 {'model__C': 10, 'scaler': 'passthrough'}
    runner up 0.757 {'model__C': 10, 'scaler': MinMaxScaler()}
    runner up 0.730 {'model__C': 0.1, 'scaler': StandardScaler()}
root, k-NN: val accuracy 0.726 {'model__metric': 'euclidean', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.726 {'model__metric': 'cosine', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.726 {'model__metric': 'cosine', 'model__n_neighbors': 1, 'sca

In [ ]:
#sklearn models, single model

print(features_used, "features,", split_used, "split")
for name in sklearn_models:
    model = build_sklearn_model(name, "chord")
    model.fit(X[train_idx], y[train_idx])
    val_accuracy = model.score(X[val_idx], y[val_idx])

    print(f"{name}: val {val_accuracy:.3f}")

wide spectrum features, random split
Logistic Regression: val 0.336
k-NN: val 0.166
SVM: val 0.485


In [47]:
#sklearn models, double model
print(features_used, "features,", split_used, "split")
for name in sklearn_models:
    root_model = build_sklearn_model(name, "root")
    quality_model = build_sklearn_model(name, "quality")
    root_model.fit(X[train_idx], targets["root"][train_idx])
    quality_model.fit(X[train_idx], targets["quality"][train_idx])

    root_pred, root_confidence = predict_sklearn_model(root_model, X[val_idx])
    quality_pred, quality_confidence = predict_sklearn_model(quality_model, X[val_idx])
    combined_pred = combine_labels(root_pred, quality_pred)

    print(name)
    print(f"root accuracy: {accuracy_score(targets['root'][val_idx], root_pred):.3f}")
    print(f"quality accuracy: {accuracy_score(targets['quality'][val_idx], quality_pred):.3f}")
    print(f"combined: {accuracy_score(y[val_idx], combined_pred):.3f}\n")

wide spectrum features, random split
Logistic Regression
root accuracy: 0.843
quality accuracy: 0.090
combined: 0.067

k-NN
root accuracy: 0.662
quality accuracy: 0.189
combined: 0.166

SVM
root accuracy: 0.886
quality accuracy: 0.313
combined: 0.278

Random Forest
root accuracy: 0.817
quality accuracy: 0.158
combined: 0.119



## For Remaining code, Evaluate the different options I have laid out for the tuning of the models. And see whats the best way to graph them :)


In [ ]:
#training code for the MLP and CNN

MAX_EPOCHS = 100 #max number of times it goes through the whole training set
PATIENCE = 30 #stops early if the validation accuracy hasnt gone up in this many epochs
SEED = 42

current_settings = {} #settings for each model and target, e.g. current_settings["MLP", "root"]
trained_models = {} #the trained models, e.g. trained_models["CNN", "chord"]

loss_function = nn.CrossEntropyLoss() #measures how wrong the predictions are


#one epoch means going through all the training clips once, in shuffled batches
def train_one_epoch(model, optimizer, inputs, answers, batch_size):
    model.train() #puts the model in training mode
    shuffled = torch.randperm(len(inputs)) #shuffles the clip numbers
    for batch in shuffled.split(batch_size): #goes through them in batches
        scores = model(inputs[batch]) #runs the batch through the model
        loss = loss_function(scores, answers[batch]) #how wrong was it
        optimizer.zero_grad() #clears the gradients from the last batch
        loss.backward() #works out the gradients
        optimizer.step() #updates the weights


#trains one model from scratch, gives back the model and its best validation accuracy
def train_net(name, target, settings):
    torch.manual_seed(SEED) #so that i get the same result every time i run it

    #scale the data (the scaler only learns from the training data)
    data = get_data(name)
    scaler = settings["scaler"]
    if scaler is not None:
        scaler = clone(scaler).fit(data[train_idx])
    inputs = get_inputs(data[train_idx], scaler)

    #pytorch doesnt understand labels like 'C:maj', it needs numbers, so each label gets a number (its position in classes)
    classes, numbers = np.unique(targets[target], return_inverse=True)
    answers = torch.tensor(numbers[train_idx]).to(device)

    #make a new model
    if name == "MLP":
        model = make_mlp(inputs.shape[1:], len(classes), settings).to(device)
    else:
        model = make_cnn(inputs.shape[1:], len(classes), settings).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=settings["lr"], weight_decay=settings["weight_decay"]) #this is what updates the weights
    net = {"model": model, "scaler": scaler, "classes": classes} #everything predict_net needs
    history = {"train_accuracy": [], "val_accuracy": []} #for the learning curves

    best_accuracy = -1
    for epoch in range(MAX_EPOCHS):
        train_one_epoch(model, optimizer, inputs, answers, settings["batch_size"])

        #check how its doing on the validation set
        labels, confidences = predict_net(net, data[val_idx])
        accuracy = accuracy_score(targets[target][val_idx], labels)

        #and on the training set, for the learning curves
        train_labels, train_confidences = predict_net(net, data[train_idx])
        history["train_accuracy"].append(accuracy_score(targets[target][train_idx], train_labels))
        history["val_accuracy"].append(accuracy)

        #save a copy of the weights if its the best so far
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())

        #stop if it hasnt improved for a while
        if epoch - best_epoch == PATIENCE:
            break

    #go back to the best weights
    model.load_state_dict(best_weights)
    history["best_epoch"] = best_epoch
    net["history"] = history
    return net, best_accuracy

In [ ]:
#the MLP, it uses X (the averaged features)
def make_mlp(input_shape, n_classes, settings):
    size_in = input_shape[0] #number of features
    hidden = settings["hidden"]
    dropout = settings["dropout"]

    #hidden layers
    layers = []
    for i in range(settings["layers"]):
        layers.append(nn.Linear(size_in, hidden))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        size_in = hidden #the next layer takes in what this one gives out

    #output layer, 1 score per class
    layers.append(nn.Linear(size_in, n_classes))

    return nn.Sequential(*layers) #runs the layers in order


mlp_settings = {"scaler": StandardScaler(), "layers": 2, "hidden": 128, "dropout": 0, "lr": 0.001, "weight_decay": 0, "batch_size": 64}

#what the tuning cell tries
mlp_options = {
    "lr": [0.0003, 0.003], #learning rate goes first since it usually matters the most
    "layers": [1, 3],
    "hidden": [64, 256, 512], #size of each hidden layer
    "dropout": [0.2, 0.4],
    "weight_decay": [0.0001, 0.001],
    "scaler": [MinMaxScaler(), Normalizer()],
    #"batch_size": [32, 128], #didnt seem worth it, it mostly does the same thing as changing lr
}

#all 3 targets start with the same settings
for target in targets:
    current_settings["MLP", target] = mlp_settings

In [ ]:
#the CNN, it uses the picture (notes x time frames)
def make_cnn(input_shape, n_classes, settings):
    notes, frames = input_shape
    kernel = settings["kernel"]
    hidden = settings["hidden"]
    dropout = settings["dropout"]

    #conv layers need a channel dimension, so (clips, notes, frames) becomes (clips, 1, notes, frames)
    layers = [nn.Unflatten(1, (1, notes))]

    #conv layers, each one has double the channels of the last
    channels_in = 1
    channels_out = settings["channels"]
    for i in range(settings["conv_layers"]):
        layers.append(nn.Conv2d(channels_in, channels_out, kernel_size=kernel, padding=kernel // 2)) #padding keeps the picture the same size

        if settings["batchnorm"]:
            layers.append(nn.BatchNorm2d(channels_out))

        layers.append(nn.ReLU())

        #only shrinks the time, i dont want to shrink the notes because then it cant tell the roots apart
        if settings["pool"] == "max":
            layers.append(nn.MaxPool2d((1, 2)))
        else:
            layers.append(nn.AvgPool2d((1, 2)))

        frames = frames // 2
        channels_in = channels_out
        channels_out = channels_out * 2

    #turns it into one long list, then the rest is basically the MLP
    layers.append(nn.Flatten())
    layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(channels_in * notes * frames, hidden))
    layers.append(nn.ReLU())
    layers.append(nn.Dropout(dropout))

    #output layer, 1 score per class
    layers.append(nn.Linear(hidden, n_classes))

    return nn.Sequential(*layers)


cnn_settings = {"scaler": None, "conv_layers": 2, "channels": 16, "kernel": 3, "pool": "max", "batchnorm": False, "hidden": 128, "dropout": 0, "lr": 0.001, "weight_decay": 0, "batch_size": 64}

#what the tuning cell tries
cnn_options = {
    "lr": [0.0003, 0.003],
    "conv_layers": [1, 3], #3 is the max since each one halves the 8 time frames
    "channels": [8, 32],
    "kernel": [5],
    "pool": ["avg"],
    "batchnorm": [True],
    "hidden": [256],
    "dropout": [0.3, 0.5],
    "weight_decay": [0.0001, 0.001],
}

#all 3 targets start with the same settings
for target in targets:
    current_settings["CNN", target] = cnn_settings

In [ ]:
#MLP and CNN tuning (this takes ages since every try trains a whole model)

tuning_log = [] #every try, for the graphs


#changes one setting at a time, and keeps the change if the validation accuracy goes up
def tune_net(name, target, options, rounds):
    best_settings = current_settings[name, target]
    net, best_accuracy = train_net(name, target, best_settings)
    tuning_log.append({"model": name, "target": target, "try": "start", "accuracy": best_accuracy, "kept": True})
    print(f"{name} {target}, starting settings: val {best_accuracy:.3f}")

    for round_number in range(rounds):
        for setting in options:
            for option in options[setting]:
                if best_settings[setting] is option:
                    continue #already using it, so no point training it again

                new_settings = best_settings.copy()
                new_settings[setting] = option
                net, accuracy = train_net(name, target, new_settings)

                kept = accuracy > best_accuracy
                if kept:
                    best_accuracy = accuracy
                    best_settings = new_settings
                tuning_log.append({"model": name, "target": target, "try": f"{setting} = {option}", "accuracy": accuracy, "kept": kept})
                print(f"    {setting} = {option}: val {accuracy:.3f}", "KEEP" if kept else "")

    print(best_settings)
    return best_settings


TUNE_MODELS = ["MLP", "CNN"]
TUNE_TARGETS = ["chord", "root", "quality"]
TUNING_ROUNDS = 1 #2 rounds can find better settings but takes twice as long

for name in TUNE_MODELS:
    for target in TUNE_TARGETS:
        if name == "MLP":
            options = mlp_options
        else:
            options = cnn_options
        current_settings[name, target] = tune_net(name, target, options, TUNING_ROUNDS)

In [ ]:
#predicts the whole chord in one go
for name in ["MLP", "CNN"]:
    net, accuracy = train_net(name, "chord", current_settings[name, "chord"])
    trained_models[name, "chord"] = net
    print(f"{name} chord: val {accuracy:.3f}")

In [ ]:
#predicts just the root
for name in ["MLP", "CNN"]:
    net, accuracy = train_net(name, "root", current_settings[name, "root"])
    trained_models[name, "root"] = net
    print(f"{name} root: val {accuracy:.3f}")

In [ ]:
#predicts just the quality
for name in ["MLP", "CNN"]:
    net, accuracy = train_net(name, "quality", current_settings[name, "quality"])
    trained_models[name, "quality"] = net
    print(f"{name} quality: val {accuracy:.3f}")

In [ ]:
#combines the root and quality predictions into full chords
for name in ["MLP", "CNN"]:
    data = get_data(name)[val_idx]
    chord_pred, chord_confidence = predict_net(trained_models[name, "chord"], data)
    root_pred, root_confidence = predict_net(trained_models[name, "root"], data)
    quality_pred, quality_confidence = predict_net(trained_models[name, "quality"], data)
    combined_pred = combine_labels(root_pred, quality_pred)

    print(name)
    print(f"root accuracy: {accuracy_score(targets['root'][val_idx], root_pred):.3f}")
    print(f"quality accuracy: {accuracy_score(targets['quality'][val_idx], quality_pred):.3f}")
    print(f"combined: {accuracy_score(y[val_idx], combined_pred):.3f}")
    print(f"whole chord in one go: {accuracy_score(y[val_idx], chord_pred):.3f}\n")